In [ ]:
import numpy as np
from scipy.stats import permutation_test

## 1. Define the populations

We simulate two populations with known conversion rates. In practice, these are unknown — here we define them to evaluate our test's ability to detect the difference.

# Statistical Power Analysis with Permutation Tests

**Goal:** Determine the minimum sample size needed to reliably detect a difference in conversion rates between two groups (A/B test).

**Key concepts:**
- **Permutation test:** Shuffles the data to estimate whether an observed difference could happen by chance.
- **p-value:** Probability of seeing a difference as extreme as the observed one, assuming no real difference exists.
- **Power:** Probability of correctly detecting a real difference. Convention: power ≥ 0.80.

In [48]:
CONVERSION_RATE_A = 0.05
CONVERSION_RATE_B = 0.15
POPULATION_SIZE = 1_000

A = np.array(
    [1] * int(POPULATION_SIZE * CONVERSION_RATE_A)
    + [0] * int(POPULATION_SIZE * (1 - CONVERSION_RATE_A))
)
B = np.array(
    [1] * int(POPULATION_SIZE * CONVERSION_RATE_B)
    + [0] * int(POPULATION_SIZE * (1 - CONVERSION_RATE_B))
)

print(f"Population A conversion rate: {A.mean():.0%}")
print(f"Population B conversion rate: {B.mean():.0%}")
print(f"True difference: {B.mean() - A.mean():.0%}")

Population A conversion rate: 5%
Population B conversion rate: 15%
True difference: 10%


## 2. Run a single experiment

Take a random sample from each population and run a permutation test to check if the observed difference is statistically significant.

In [49]:
ALPHA = 0.05
SAMPLE_SIZE = 1_000

rng = np.random.default_rng(seed=1997)


def statistic(x, y, axis=0):
    return np.mean(x, axis=axis) - np.mean(y, axis=axis)


sample_A = rng.choice(A, size=SAMPLE_SIZE, replace=True)
sample_B = rng.choice(B, size=SAMPLE_SIZE, replace=True)

result = permutation_test(
    (sample_A, sample_B),
    statistic,
    vectorized=True,
    n_resamples=10_000,
    alternative="two-sided",
)

print(f"Sample A conversion: {sample_A.mean():.2%}")
print(f"Sample B conversion: {sample_B.mean():.2%}")
print(f"Observed difference: {result.statistic:.4%}")
print(f"p-value: {result.pvalue:.4f}")
print(f"Significant at alpha={ALPHA}: {'Yes' if result.pvalue < ALPHA else 'No'}")

Sample A conversion: 5.70%
Sample B conversion: 16.10%
Observed difference: -10.4000%
p-value: 0.0002
Significant at alpha=0.05: Yes


## 3. Estimate power for a fixed sample size

One experiment isn't enough to know if our design is reliable. We repeat the experiment many times and measure how often we correctly detect the difference.

In [52]:
N_EXPERIMENTS = 500
SAMPLE_SIZE = 160
significant_count = 0

for _ in range(N_EXPERIMENTS):
    sample_A = rng.choice(A, size=SAMPLE_SIZE, replace=True)
    sample_B = rng.choice(B, size=SAMPLE_SIZE, replace=True)

    result = permutation_test(
        (sample_A, sample_B),
        statistic,
        vectorized=True,
        n_resamples=1_000,
        alternative="two-sided",
    )

    if result.pvalue < ALPHA:
        significant_count += 1

power = significant_count / N_EXPERIMENTS
print(f"Sample size: {SAMPLE_SIZE}")
print(
    f"Power: {power:.2%} ({significant_count}/{N_EXPERIMENTS} experiments detected the difference)"
)

Sample size: 160
Power: 82.00% (410/500 experiments detected the difference)


## 4. Find the minimum sample size

We repeat the power estimation for different sample sizes to find the smallest one that achieves power ≥ 0.80.

In [58]:
N_EXPERIMENTS = 500
SAMPLE_SIZES = [n for n in range(140, 201, 10)]
TARGET_POWER = 0.80

results = []

for n in SAMPLE_SIZES:
    significant_count = 0

    for _ in range(N_EXPERIMENTS):
        sample_A = rng.choice(A, size=n, replace=True)
        sample_B = rng.choice(B, size=n, replace=True)

        result = permutation_test(
            (sample_A, sample_B),
            statistic,
            vectorized=True,
            n_resamples=500,
            alternative="two-sided",
        )

        if result.pvalue < ALPHA:
            significant_count += 1

    power = significant_count / N_EXPERIMENTS
    results.append({"n": n, "power": power})
    reach_target = power >= TARGET_POWER
    marker = " ✓" if reach_target else ""
    print(f"n={n:>5}, Power: {power:.2%}{marker}")

    if reach_target:
        print(f"Reached target power of {TARGET_POWER:.0%} at sample size n={n}")
        break

n=  140, Power: 77.40%
n=  150, Power: 80.60% ✓
Reached target power of 80% at sample size n=150


## Conclusion

With a true difference of 10 percentage points (5% vs 15%), we need approximately **180 users per group** to detect the difference with ≥ 80% power at α = 0.05.